## Repaired points

In [17]:
from input_space.divide_sets import (
    make_root_region, 
    make_bounded_model, 
    compute_margin_lower_bounds_reuse,
    analyze_region,
    split_region,
    explore_input_space,)
from input_space.generate_input import ( repair_points )
from experiments import mnist
import sytorch as st

device = 'cpu'
dtype = st.float64
dnn = mnist.model('9x100').to(dtype=dtype).to(device=device)
target_label = 8
repaired_indices, repaired_points, base_points = repair_points(
    repair_label=target_label, 
    dnn=dnn, device=device, dtype=dtype)


x_repaired = repaired_points.images[0].unsqueeze(0).to(device=device, dtype=dtype)  # shape: (1, 784)
x_base = base_points.images[0].unsqueeze(0).to(device=device, dtype=dtype)  # shape: (1, 784)
y_label = repaired_points.labels[0].item()        # scalar
print(f"repaired point label: {y_label}")

print(f'len(repaired_points.images): {len(repaired_points.images)}')
print(f'len(base_points.images): {len(base_points.images)}')
print(repaired_points.indices[:10])
print(base_points.indices[:10])

root_region_re = make_root_region(x=x_repaired, eps=0.01, target_label=target_label)
root_region_ba = make_root_region(x=x_base, eps=0.01, target_label=target_label)

in_shape = x_repaired.shape
root_bounded_model = make_bounded_model(dnn, in_shape, device)

lb_re, ub_re = root_region_re.lb, root_region_re.ub
lb_ba, ub_ba = root_region_ba.lb, root_region_ba.ub
out_re = dnn(x_repaired)
out_ba = dnn(x_base)
out_label_re = out_re.argmax(dim=1).item()
out_label_ba = out_ba.argmax(dim=1).item()
num_classes = out_re.shape[1]
bounding_method = 'CROWN'
margin_lbs, other_labels = compute_margin_lower_bounds_reuse(root_bounded_model, lb_re, ub_re, target_label, num_classes, bounding_method)

root_region_re = analyze_region(dnn, root_bounded_model, root_region_re, num_classes, bounding_method)
root_region_ba = analyze_region(dnn, root_bounded_model, root_region_ba, num_classes, bounding_method)

print("repaired region status:", root_region_re.status)
print("base region status:", root_region_ba.status)

repaired point label: 8
len(repaired_points.images): 444
len(base_points.images): 444
[177, 181, 184, 232, 257, 260, 266, 268, 290, 338]
[177, 181, 184, 232, 257, 260, 266, 268, 290, 338]
repaired region status: negative
base region status: positive


## Network Bound (Backsubstitution + Alpha-CROWN)

In [1]:
from input_space.generate_input import repair_points
from experiments import mnist
import sytorch as st

import torch
import torch.nn as nn

from network_bound.a_crown import AlphaCROWNSequential

device = 'cpu'
dtype = st.float64

# Load network
dnn = mnist.model('9x100').to(dtype=dtype).to(device=device)

target_label = 8
repaired_indices, repaired_points, base_points = repair_points(
    repair_label=target_label,
    dnn=dnn,
    device=device,
    dtype=dtype,
)

# One input image
x = base_points.images[0].unsqueeze(0).to(device=device, dtype=dtype)   # (1, 784)

# ------------------------------------------------------------
# 1. Check prediction
# ------------------------------------------------------------
with torch.no_grad():
    logits = dnn(x)
    pred_label = logits.argmax(dim=1).item()
    print("logits:", logits)
    print("pred_label:", pred_label)

# ------------------------------------------------------------
# 2. Create alpha-CROWN bounder
# ------------------------------------------------------------
bounder = AlphaCROWNSequential(dnn)

# ------------------------------------------------------------
# 3. Choose which label to certify
# ------------------------------------------------------------
# Usually for local robustness, certify the predicted label.
cert_label = pred_label

# If instead you want to certify label 8 specifically, use:
# cert_label = 8

# ------------------------------------------------------------
# 4. Build one-vs-all specification matrix C
#    Each row is e_cert - e_j
# ------------------------------------------------------------
out_dim = logits.shape[1]
specs = []

for j in range(out_dim):
    if j == cert_label:
        continue
    c = torch.zeros(out_dim, dtype=dtype, device=device)
    c[cert_label] = 1.0
    c[j] = -1.0
    specs.append(c)

C = torch.stack(specs, dim=0).unsqueeze(0)   # shape: (1, 9, 10) for MNIST
print("C.shape:", C.shape)

# ------------------------------------------------------------
# 5. Compute alpha-CROWN bounds
# ------------------------------------------------------------
eps = 0.05

result = bounder.compute_bounds(
    x_center=x,
    eps=eps,
    C=C,
    alpha_steps=50,
    alpha_lr=1e-1,
    optimize_alpha=True,
    verbose=True,
)

# ------------------------------------------------------------
# 6. Inspect results
# ------------------------------------------------------------
print("\n=== Final bounds ===")
print("lower shape:", result.lower.shape)   # (B, M)
print("upper shape:", result.upper.shape)
print("lower:", result.lower)
print("upper:", result.upper)

# Each lower bound corresponds to:
#   f_cert(x') - f_j(x')   over all ||x' - x||_inf <= eps
#
# If all lower bounds > 0, then cert_label is provably robust in that box.
is_certified = (result.lower[0] > 0).all().item()
print(f"\nCertified for label {cert_label} with eps={eps}? ->", is_certified)

# ------------------------------------------------------------
# 7. See which class margins are hardest
# ------------------------------------------------------------
margin_idx = 0
for j in range(out_dim):
    if j == cert_label:
        continue
    lb = result.lower[0, margin_idx].item()
    ub = result.upper[0, margin_idx].item()
    print(f"class {cert_label} vs {j}: lower={lb:.6f}, upper={ub:.6f}")
    margin_idx += 1

# ------------------------------------------------------------
# 8. Inspect backpropagated dual coefficients A
# ------------------------------------------------------------
print("\n=== A tensors (lower bound pass) ===")
for k, A in sorted(result.A_dict_lower.items(), key=lambda kv: kv[0]):
    print(f"layer {k}: shape = {tuple(A.shape)}")

# Input-level dual coefficients
A_input = result.A_dict_lower[-1]   # shape: (1, M, 784)
print("\nA_input.shape:", A_input.shape)

# Example: the dual coefficients for the first spec row
first_spec_A = A_input[0, 0]        # shape: (784,)
print("first_spec_A[:20] =", first_spec_A[:20])

# ------------------------------------------------------------
# 9. Inspect optimized alpha values
# ------------------------------------------------------------
print("\n=== alpha tensors ===")
for k, alpha in sorted(result.alpha_dict.items()):
    print(f"ReLU after linear layer {k}: shape = {tuple(alpha.shape)}")
    print(alpha[0, :10])   # show first 10 neurons

/Users/kotafukuda/Documents/Project_univ/repair/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/kotafukuda/Documents/Project_univ/repair/.venv/lib/python3.12/site-packages/torch/functional.py:512: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:3588.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


logits: tensor([[-1.1645, -1.4362, -0.0899,  0.9207, -2.5611,  1.6995, -0.9881, -1.5938,
          6.7781,  0.3418]], dtype=torch.float64)
pred_label: 8
C.shape: torch.Size([1, 9, 10])
[alpha step 001] mean lower = -100700.228116
[alpha step 010] mean lower = -83296.461411
[alpha step 020] mean lower = -83265.295190
[alpha step 030] mean lower = -83262.573453
[alpha step 040] mean lower = -83261.819621
[alpha step 050] mean lower = -83261.231037

=== Final bounds ===
lower shape: torch.Size([1, 9])
upper shape: torch.Size([1, 9])
lower: tensor([[-59891.1003, -73364.7741, -89154.5717, -85831.1621, -96798.4289,
         -73616.6546, -81667.2416, -96787.0344, -92240.7185]],
       dtype=torch.float64)
upper: tensor([[116206.3801, 122630.9885,  90489.0347,  84684.8067, 120905.9597,
          81967.7691, 109233.2343, 106221.0744, 115316.5983]],
       dtype=torch.float64)

Certified for label 8 with eps=0.05? -> False
class 8 vs 0: lower=-59891.100296, upper=116206.380086
class 8 vs 1: lowe

In [6]:
from input_space.generate_input import repair_points
from experiments import mnist
import sytorch as st

import torch
import torch.nn as nn

device = 'cpu'
dtype = st.float64

# Load network
dnn = mnist.model('9x100').to(dtype=dtype).to(device=device)

target_label = 8
repaired_indices, repaired_points, base_points = repair_points(
    repair_label=target_label,
    dnn=dnn,
    device=device,
    dtype=dtype,
)

# One input image
with torch.no_grad():
    x = base_points.images[0].unsqueeze(0).to(device=device, dtype=dtype)   # (1, 784)
    eps = 0.05
    lb_inp = (x - eps).squeeze(0)   # shape: (784,)
    ub_inp = (x + eps).squeeze(0)   # shape: (784,)

from network_bound.backsubstitution import IndividualBounds

bounder = IndividualBounds(
    net=dnn,
    lb_inp=lb_inp,
    ub_inp=ub_inp,
    device=device,
)

lbs, ubs = bounder.run(
    optimize_alpha=True,
    alpha_steps=20,
    alpha_lr=1e-1,
    save_coeffs=True,
    verbose=True,
)

print(f"len(lbs): {len(lbs)}, len(ubs): {len(ubs)}")
print("final lb:", lbs[-1])
print("final ub:", ubs[-1])
print("saved coefficient keys:", list(bounder.saved_coeffs.keys())[:10])

[alpha step 001] objective = -16482.238721
[alpha step 010] objective = -3066.877969
[alpha step 020] objective = -3040.363595
len(lbs): 18, len(ubs): 18
final lb: tensor([-2372.6010, -2541.8448, -1437.3799, -1656.7426, -2099.8819, -1470.5015,
        -2205.7208, -1905.1248, -1625.4529, -2207.9872], dtype=torch.float64,
       grad_fn=<AddBackward0>)
final ub: tensor([1397.2946, 1948.9644, 2182.5340, 2481.7345, 2204.6395, 1974.8423,
        2062.8044, 2466.8537, 2430.5899, 2407.6377], dtype=torch.float64,
       grad_fn=<AddBackward0>)
saved coefficient keys: [(0, 0), (1, 1), (1, 0), (2, 2), (2, 1), (2, 0), (3, 3), (3, 2), (3, 1), (3, 0)]


In [7]:
dnn

Sequential(
  (0): Linear(in_features=784, out_features=100, bias=True)
  (1): ReLU()
  (2): Linear(in_features=100, out_features=100, bias=True)
  (3): ReLU()
  (4): Linear(in_features=100, out_features=100, bias=True)
  (5): ReLU()
  (6): Linear(in_features=100, out_features=100, bias=True)
  (7): ReLU()
  (8): Linear(in_features=100, out_features=100, bias=True)
  (9): ReLU()
  (10): Linear(in_features=100, out_features=100, bias=True)
  (11): ReLU()
  (12): Linear(in_features=100, out_features=100, bias=True)
  (13): ReLU()
  (14): Linear(in_features=100, out_features=100, bias=True)
  (15): ReLU()
  (16): Linear(in_features=100, out_features=10, bias=True)
)